# CA interaction computation

This notebook performs the calculations. Adjacent salt transitions are the default because their structural ensembles overlap more strongly. The single-reference calculation remains available as a comparison mode, and the lowest-to-highest transition can be run as a closure check. The fitted spectrum and anisotropy tensor of every source and target state are held fixed during each comparison.

In [1]:
from __future__ import annotations

import csv
from pathlib import Path

import numpy as np

import cf_ca_interaction as interaction

HERE = Path.cwd().resolve()
if not (HERE / "cf_ca_interaction.py").exists():
    raise RuntimeError("Run this notebook from smpl/curvefit")
PROJECT_ROOT = HERE.parent.parent

# 1. SETTINGS THAT MUST MATCH BETWEEN COMPUTE AND RESULTS.
# Copy this whole block into the results notebook before loading saved results.
# Scientific default: anisotropic geometry with adjacent-state comparisons.
FIT_SOURCE = "aniso"  # "iso" or "aniso"
COMPARISON_MODE = "adjacent"  # "adjacent" or "single_reference"
REFERENCE_CONDITION = "zero"  # "zero" or "highest"
ANISOTROPIC_GEOMETRY = True  # True for anisotropic geometry; False for isotropic geometry.
LOCAL_EXPANSION_RANGE_MULTIPLIER = 1.0  # ell = this value / k_eff; the note uses 1.0.
# END MATCHED SETTINGS

if COMPARISON_MODE not in {"adjacent", "single_reference"}:
    raise ValueError("COMPARISON_MODE must be 'adjacent' or 'single_reference'")
if REFERENCE_CONDITION not in {"zero", "highest"}:
    raise ValueError("REFERENCE_CONDITION must be 'zero' or 'highest'")

# Input locations and fixed numerical settings.
ANISOTROPY_AXIS = 2
ANISOTROPY_H_BY_SALT = None  # Optional {concentration_mM: 3x3 H}; None uses fitted stretch_ratio.
FIT_SOURCES = {
    "iso": {"input_dir": HERE / "output" / "ca_iso", "pattern": "ca_*mM_gamma_radial_unguided_gamma_radial_isotropic_fit_parameters.csv"},
    "aniso": {"input_dir": HERE / "output" / "ca_aniso" / "gamma_OH2", "pattern": "ca_*mM_gamma_radial_unguided_gamma_radial_full_geom_OH2_fit_parameters.csv"},
}
LOCAL_SAMPLE_POWER = 15
LOCAL_RANDOM_SEED = 73129
RECONSTRUCTION_KS_WARNING = 0.08
PROJECTION_UNEXPLAINED_WARNING = 0.20
LOCAL_K2_HISTOGRAM_RANGE = (0.0, 5.0)  # Ignore extreme outliers when defining display bins.
LOCAL_K4_HISTOGRAM_RANGE = (-0.25, 0.25)  # Focus on the resolved central signed distribution.
LOCAL_K4_HISTOGRAM_BINS = 180  # Finer bins for the narrow fourth-order range.

# 2. SET TRUE WHEN NEW INPUT DATA OR FIT RESULTS ARE LOADED.
RUN_RANDOMWAVE_SAMPLING = True  # Primary calculation from conditioned random-wave samples.
BUILD_REFERENCE_TRACE = True  # Needed only for an explicitly enabled trace-based check.
BUILD_TARGET_TRACES = True  # Needed only for an explicitly enabled trace-based check.

# 3. SET TRUE ONLY WHEN THE CORRESPONDING ADVANCED CHECK IS REQUIRED.
RUN_FLEXIBLE_DIAGNOSTIC = True  # Not required for the contour-integrated test.
RUN_EXPANSION_RANGE_SENSITIVITY = False
RUN_REPLICATE_UNCERTAINTY = False
RUN_NONLOCAL_DEPENDENCE = True  # If enabled, test every salt condition.
RUN_NONLOCAL_CORRECTION = True  # If any condition needs correction, correct every comparison pair.
RUN_CONTOUR_VALIDATION = False  # Optional trace-based finite-size/resolution check; never used for the primary fit.
RUN_TRACE_CONVERGENCE = False
RUN_LONG_BASELINE_CLOSURE = False  # Compare endpoint inference with the sum of adjacent steps.

# Numerical settings used only by the advanced checks above.
REPLICATE_SAMPLE_POWER = 13
REPLICATE_BOOTSTRAP_SAMPLES = 400
NONLOCAL_KERNEL_MODE = "auto"
NONLOCAL_TWO_RANGES_KREF = (0.5, 2.0)
NONLOCAL_SHAPE_CORRECTION_OVERRIDE = None
NONLOCAL_MAX_ITERATIONS = 5
NONLOCAL_DAMPING = 0.5
CONTOUR_BLOCK_RANGE_MULTIPLIER = 8.0  # Equal block length in units of 1/k_source.
CONTOUR_SUPPORT_QUANTILES = (0.02, 0.98)  # Exclude traced derivative singularities from fitting.
CLOSURE_UNCERTAINTY_SAMPLES = 400
REFERENCE_TAG = "0mM" if REFERENCE_CONDITION == "zero" else "highest_salt"
COMPARISON_TAG = "_adjacent" if COMPARISON_MODE == "adjacent" else ""  # Keep the former single-reference path compatible.
OUTPUT_SUFFIX = "_geometry_aniso" if ANISOTROPIC_GEOMETRY else ""
OUTPUT_DIR = HERE / "output" / f"ca_interaction_refactored_{REFERENCE_TAG}_{FIT_SOURCE}{COMPARISON_TAG}{OUTPUT_SUFFIX}"
ISOTROPIC_TRACE_DIR = HERE / "output" / f"ca_interaction_refactored_{REFERENCE_TAG}_{FIT_SOURCE}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRACE_CONFIG = interaction.TraceConfig(grid_size=64, num_blocks=1, num_modes=256, trace_k0=5.0, random_seed=894894, seed_spacing=0.35, q_spacing=0.08)

## Prepare the local samples and their common range

In [2]:
source = FIT_SOURCES[FIT_SOURCE]
fit_series = interaction.discover_fit_spectra(source["input_dir"], source["pattern"])
spectrum_by_salt = {row.concentration_mM: row for row in fit_series}
reference_spectrum = interaction.select_reference_spectrum(fit_series, REFERENCE_CONDITION)
REFERENCE_SALT = reference_spectrum.concentration_mM
REFERENCE_K_EFF = reference_spectrum.k_eff  # Fixed reporting scale, not the scale used for every adjacent fit.
comparison_pairs = interaction.interaction_comparison_pairs(fit_series, COMPARISON_MODE, REFERENCE_CONDITION)
pair_by_target = {pair.target_concentration_mM: pair for pair in comparison_pairs}
fit_input_rows = [{"concentration_mM": row.concentration_mM, "reference_concentration_mM": REFERENCE_SALT, "reference_condition": REFERENCE_CONDITION, "comparison_mode": COMPARISON_MODE, "k_eff": row.k_eff, "mean_k": row.mean_k, "r_sigma_k": row.r_sigma_k, "stretch_ignored": row.stretch_ignored, "source": str(row.source)} for row in fit_series]
with (OUTPUT_DIR / "radial_fit_inputs.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(fit_input_rows[0]))
    writer.writeheader(); writer.writerows(fit_input_rows)
if ANISOTROPIC_GEOMETRY:
    hencky_by_salt = (interaction.fitted_hencky_tensors(fit_series, ANISOTROPY_AXIS) if ANISOTROPY_H_BY_SALT is None else {float(salt): interaction.validate_hencky_tensor(H) for salt,H in ANISOTROPY_H_BY_SALT.items()})
else:
    hencky_by_salt = None
local_samples = interaction.sample_local_geometry_series(
    fit_series, sample_power=LOCAL_SAMPLE_POWER, seed=LOCAL_RANDOM_SEED,
    aniso=ANISOTROPIC_GEOMETRY, H_by_concentration=hencky_by_salt,
)
if ANISOTROPIC_GEOMETRY:
    anisotropy_rows = []
    for row in fit_series:
        H = hencky_by_salt[row.concentration_mM]; F = interaction.hencky_stretch_tensor(H)
        h_principal = np.linalg.eigvalsh(H); lambda_principal = np.exp(h_principal)
        anisotropy_rows.append({"concentration_mM": row.concentration_mM, "stretch_ratio": row.stretch_ignored, "anisotropy_axis": ANISOTROPY_AXIS, "h1": h_principal[0], "h2": h_principal[1], "h3": h_principal[2], "lambda1": lambda_principal[0], "lambda2": lambda_principal[1], "lambda3": lambda_principal[2], "H_xx": H[0,0], "H_yy": H[1,1], "H_zz": H[2,2], "H_xy": H[0,1], "H_xz": H[0,2], "H_yz": H[1,2], "F_xx": F[0,0], "F_yy": F[1,1], "F_zz": F[2,2], "F_xy": F[0,1], "F_xz": F[0,2], "F_yz": F[1,2]})
    with (OUTPUT_DIR / "anisotropy_inputs.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(anisotropy_rows[0])); writer.writeheader(); writer.writerows(anisotropy_rows)
reference_sample = local_samples[REFERENCE_SALT]
LOCAL_EXPANSION_RANGE = LOCAL_EXPANSION_RANGE_MULTIPLIER / REFERENCE_K_EFF
rng = np.random.default_rng(9123)
train_mask = rng.random(len(reference_sample.weights)) < 0.67
test_mask = ~train_mask
fit_results = {}
heldout_results = {}
pair_context_by_target = {}
for pair in comparison_pairs:
    source_spectrum = spectrum_by_salt[pair.source_concentration_mM]
    source_sample = local_samples[pair.source_concentration_mM]
    target_sample = local_samples[pair.target_concentration_mM]
    source_k_eff = source_spectrum.k_eff
    source_expansion_range = LOCAL_EXPANSION_RANGE_MULTIPLIER / source_k_eff
    result = interaction.fit_local_cell_reweighting(source_sample, target_sample, source_k_eff, cell_length=source_expansion_range)
    heldout = interaction.fit_local_cell_reweighting(source_sample, target_sample, source_k_eff, cell_length=source_expansion_range, reference_mask=train_mask, target_mask=train_mask)
    fit_results[pair.target_concentration_mM] = result
    heldout_results[pair.target_concentration_mM] = heldout
    pair_context_by_target[pair.target_concentration_mM] = {"pair": pair, "source_spectrum": source_spectrum, "source_sample": source_sample, "target_sample": target_sample, "source_k_eff": source_k_eff, "expansion_range": source_expansion_range}
print(f"prepared {len(comparison_pairs)} {COMPARISON_MODE} comparisons; cumulative reference={REFERENCE_SALT:g} mM; anisotropic geometry={ANISOTROPIC_GEOMETRY}")

prepared 7 adjacent comparisons; cumulative reference=0 mM; anisotropic geometry=True


## Infer $\Delta c_2,\Delta c_4$ and check the distributions

The reported coefficients are fitted directly with the linked $K_2,K_4$ Hamiltonian. An independent fourth-order fit and the flexible polynomial are saved only as diagnostics when the linked model cannot reproduce the measured distributions.

In [3]:
if RUN_RANDOMWAVE_SAMPLING:
    reconstruction_payload = {}
    reconstruction_quality_rows = []
    linked_fit_results = {}
    independent_fit_results = {}
    anchor_salt = min(spectrum_by_salt) if COMPARISON_MODE == "adjacent" else REFERENCE_SALT
    anchor_metadata = {"source_concentration_mM": anchor_salt, "target_concentration_mM": anchor_salt, "pair_id": f"{anchor_salt:g}mM_anchor", "step_index": -1, "comparison_mode": COMPARISON_MODE, "consistency_check": False}
    coefficient_rows = [{**anchor_metadata, "concentration_mM": anchor_salt, "delta_c2_kref": 0.0, "delta_c2_kref_se_naive": 0.0, "delta_c4_kref3": 0.0, "delta_c4_kref3_se_naive": 0.0, "delta_c2_ksource": 0.0, "delta_c4_ksource3": 0.0, "source_k_eff": spectrum_by_salt[anchor_salt].k_eff, "randomwave_target_K2_over_ksource": np.nan, "randomwave_fitted_K2_over_ksource": np.nan, "randomwave_target_K4_over_ksource3": np.nan, "randomwave_fitted_K4_over_ksource3": np.nan, "cov_c2_c2": 0.0, "cov_c2_c4": 0.0, "cov_c4_c4": 0.0, "moment_residual_c2_feature": 0.0, "moment_residual_c4_feature": 0.0, "reference_effective_fraction": 1.0, "target_mass_retained": 1.0, "projection_unexplained_fraction": 0.0, "projection_effective_count": np.nan, "local_expansion_range_times_kref": LOCAL_EXPANSION_RANGE_MULTIPLIER, "local_cell_length_times_kref": LOCAL_EXPANSION_RANGE_MULTIPLIER, "inference_method": "linked_maximum_entropy"}]
    independent_rows = [{**anchor_metadata, "concentration_mM": anchor_salt, "a2_kref": 0.0, "a_kappa4_kref3": 0.0, "a_kappaprime2_kref3": 0.0, "a_kappatau2_kref3": 0.0, "linked_delta_c4_kref3": 0.0, "residual_kappa4": 0.0, "residual_kappaprime2": 0.0, "residual_kappatau2": 0.0, "residual_fraction": 0.0}]
    invariant_moment_rows = []
    structure_correlation_rows = []
    local_structure_histogram_rows = []
    moment_names = ("kappa2", "kappa4", "kappaprime2", "kappatau2")
    reference_invariants = interaction.local_invariant_observables(reference_sample, REFERENCE_K_EFF)
    invariant_bounds = np.vstack([interaction.weighted_quantile(reference_invariants[:,index], [0.002,0.998], reference_sample.weights) for index in range(4)])
    for spectrum in fit_series:
        sample = local_samples[spectrum.concentration_mM]
        invariants = interaction.local_invariant_observables(sample, REFERENCE_K_EFF)
        invariant_keep = np.all((invariants[:,:4] >= invariant_bounds[:,0]) & (invariants[:,:4] <= invariant_bounds[:,1]), axis=1)
        moment_mean, moment_std = interaction.weighted_mean_and_std(invariants[invariant_keep], sample.weights[invariant_keep])
        full_mean, full_std = interaction.weighted_mean_and_std(invariants, sample.weights)
        invariant_moment_rows.append({"concentration_mM": spectrum.concentration_mM, "reference_support_mass": float(sample.weights[invariant_keep].sum()/sample.weights.sum()), **{f"mean_{name}": value for name,value in zip(moment_names,moment_mean)}, **{f"std_{name}": value for name,value in zip(moment_names,moment_std)}, **{f"full_mean_{name}": value for name,value in zip(moment_names,full_mean)}, **{f"full_std_{name}": value for name,value in zip(moment_names,full_std)}})
        local_state = np.column_stack((sample.kappa/REFERENCE_K_EFF, sample.kappa_prime/REFERENCE_K_EFF**2, sample.kappa_tau/REFERENCE_K_EFF**2))
        magnitude_correlation = interaction.weighted_correlation_matrix(np.log1p(local_state[invariant_keep]**2), sample.weights[invariant_keep])
        structure_correlation_rows.append({"concentration_mM": spectrum.concentration_mM, "corr_kappa_kappa_prime": magnitude_correlation[0,1], "corr_kappa_kappa_tau": magnitude_correlation[0,2], "corr_kappa_prime_kappa_tau": magnitude_correlation[1,2], "definition": "weighted correlation of log(1+normalized variable squared) on common reference support"})
    structure_values = {"kappa": {spectrum.concentration_mM: local_samples[spectrum.concentration_mM].kappa/REFERENCE_K_EFF for spectrum in fit_series}, "kappa_prime": {spectrum.concentration_mM: local_samples[spectrum.concentration_mM].kappa_prime/REFERENCE_K_EFF**2 for spectrum in fit_series}, "kappa_tau": {spectrum.concentration_mM: local_samples[spectrum.concentration_mM].kappa_tau/REFERENCE_K_EFF**2 for spectrum in fit_series}}
    structure_ranges = {}
    for name, values_by_salt in structure_values.items():
        bounds = np.asarray([interaction.weighted_quantile(values_by_salt[spectrum.concentration_mM], [0.005,0.995], local_samples[spectrum.concentration_mM].weights) for spectrum in fit_series])
        if name == "kappa": structure_ranges[name] = (0.0, float(np.max(bounds[:,1])))
        else:
            limit = float(np.max(np.abs(bounds))); structure_ranges[name] = (-limit,limit)
        for spectrum in fit_series:
            concentration = spectrum.concentration_mM; sample = local_samples[concentration]
            density, edges = np.histogram(values_by_salt[concentration], bins=100, range=structure_ranges[name], weights=sample.weights, density=True)
            centers = 0.5*(edges[:-1]+edges[1:])
            local_structure_histogram_rows.extend({"concentration_mM": concentration, "variable": name, "coordinate": coordinate, "density": value} for coordinate,value in zip(centers,density))
    for concentration, logistic_start in heldout_results.items():
        context = pair_context_by_target[concentration]
        pair = context["pair"]
        source_sample = context["source_sample"]
        source_k_eff = context["source_k_eff"]
        source_expansion_range = context["expansion_range"]
        target = local_samples[concentration]
        initial = np.array([logistic_start.delta_c2_kref, logistic_start.delta_c4_kref3])
        full_initial = np.array([fit_results[concentration].delta_c2_kref, fit_results[concentration].delta_c4_kref3])
        linked = interaction.fit_linked_maximum_entropy(source_sample, target, source_k_eff, cell_length=source_expansion_range, initial=full_initial)
        linked_train = interaction.fit_linked_maximum_entropy(source_sample, target, source_k_eff, cell_length=source_expansion_range, reference_mask=train_mask, target_mask=train_mask, initial=initial)
        independent = interaction.fit_independent_maximum_entropy(source_sample, target, source_k_eff, cell_length=source_expansion_range)
        independent_train = interaction.fit_independent_maximum_entropy(source_sample, target, source_k_eff, cell_length=source_expansion_range, reference_mask=train_mask, target_mask=train_mask)
        linked_fit_results[concentration] = linked
        independent_fit_results[concentration] = independent
        linked_reference_support = np.all((interaction.local_cell_features(source_sample, source_k_eff, source_expansion_range) >= linked.feature_bounds[:,0]) & (interaction.local_cell_features(source_sample, source_k_eff, source_expansion_range) <= linked.feature_bounds[:,1]), axis=1)
        linked_target_support = np.all((interaction.local_cell_features(target, source_k_eff, source_expansion_range) >= linked.feature_bounds[:,0]) & (interaction.local_cell_features(target, source_k_eff, source_expansion_range) <= linked.feature_bounds[:,1]), axis=1)
        independent_reference_features = interaction.independent_local_cell_features(source_sample, source_k_eff, source_expansion_range)
        independent_target_features = interaction.independent_local_cell_features(target, source_k_eff, source_expansion_range)
        independent_reference_support = np.all((independent_reference_features >= independent.feature_bounds[:,0]) & (independent_reference_features <= independent.feature_bounds[:,1]), axis=1)
        independent_target_support = np.all((independent_target_features >= independent.feature_bounds[:,0]) & (independent_target_features <= independent.feature_bounds[:,1]), axis=1)
        source_evaluation = test_mask & linked_reference_support & independent_reference_support
        target_evaluation = test_mask & linked_target_support & independent_target_support
        linked_weights = interaction.maximum_entropy_reweighted_weights(source_sample, linked_train, source_k_eff, source_expansion_range, mask=source_evaluation)
        independent_weights = interaction.maximum_entropy_reweighted_weights(source_sample, independent_train, source_k_eff, source_expansion_range, independent=True, mask=source_evaluation)
        flexible = interaction.fit_flexible_density_ratio(source_sample, target, source_k_eff, degree=4, reference_mask=train_mask & linked_reference_support, target_mask=train_mask & linked_target_support) if RUN_FLEXIBLE_DIAGNOSTIC else None
        projection = interaction.project_flexible_fit_to_linked_coefficients(source_sample, flexible, source_k_eff, cell_length=source_expansion_range, mask=train_mask & linked_reference_support) if flexible is not None else None
        flexible_full = None
        if flexible is not None:
            flexible_log_ratio = flexible.log_ratio(source_sample.state, source_k_eff)
            flexible_log_ratio -= np.max(flexible_log_ratio[source_evaluation])
            flexible_full = source_sample.weights * np.exp(np.maximum(flexible_log_ratio, -60.0))
        projection_unexplained = np.nan if projection is None else projection.unexplained_fraction
        if projection is not None and projection.unexplained_fraction > PROJECTION_UNEXPLAINED_WARNING:
            print(f"WARNING {pair.pair_id}: K2,K4 leave {projection.unexplained_fraction:.1%} of the fitted log-density variation unexplained")
        reported_coefficients = interaction.rescale_dimensionless_coefficient_change(linked.coefficients, source_k_eff, REFERENCE_K_EFF)
        reported_covariance = interaction.rescale_coefficient_covariance(linked.covariance, source_k_eff, REFERENCE_K_EFF)
        linked_errors = np.sqrt(np.maximum(np.diag(reported_covariance), 0.0))
        coefficient_rows.append({**pair.metadata, "concentration_mM": concentration, "delta_c2_kref": reported_coefficients[0], "delta_c2_kref_se_naive": linked_errors[0], "delta_c4_kref3": reported_coefficients[1], "delta_c4_kref3_se_naive": linked_errors[1], "delta_c2_ksource": linked.delta_c2_kref, "delta_c4_ksource3": linked.delta_c4_kref3, "source_k_eff": source_k_eff, "randomwave_target_K2_over_ksource": linked.target_moments[0], "randomwave_fitted_K2_over_ksource": linked.fitted_moments[0], "randomwave_target_K4_over_ksource3": linked.target_moments[1], "randomwave_fitted_K4_over_ksource3": linked.fitted_moments[1], "cov_c2_c2": reported_covariance[0,0], "cov_c2_c4": reported_covariance[0,1], "cov_c4_c4": reported_covariance[1,1], "moment_residual_c2_feature": linked.moment_residual[0], "moment_residual_c4_feature": linked.moment_residual[1], "reference_effective_fraction": linked.reference_effective_fraction, "target_mass_retained": linked.target_mass_retained, "projection_unexplained_fraction": projection_unexplained, "projection_effective_count": np.nan if projection is None else projection.effective_count, "local_expansion_range_times_kref": LOCAL_EXPANSION_RANGE_MULTIPLIER, "local_cell_length_times_kref": LOCAL_EXPANSION_RANGE_MULTIPLIER, "inference_method": "linked_maximum_entropy"})
        independent_scaled = independent.coefficients * np.array([REFERENCE_K_EFF/source_k_eff, (REFERENCE_K_EFF/source_k_eff)**3, (REFERENCE_K_EFF/source_k_eff)**3, (REFERENCE_K_EFF/source_k_eff)**3])
        independent_c4, independent_residual, independent_residual_fraction = interaction.decompose_independent_fourth_coefficients(independent_scaled)
        independent_rows.append({**pair.metadata, "concentration_mM": concentration, "a2_kref": independent_scaled[0], "a_kappa4_kref3": independent_scaled[1], "a_kappaprime2_kref3": independent_scaled[2], "a_kappatau2_kref3": independent_scaled[3], "linked_delta_c4_kref3": independent_c4, "residual_kappa4": independent_residual[0], "residual_kappaprime2": independent_residual[1], "residual_kappatau2": independent_residual[2], "residual_fraction": independent_residual_fraction})
        variable_payload = {}
        quality = {**pair.metadata, "concentration_mM": concentration, "fit_success": linked_train.success, "independent_fit_success": independent_train.success, "flexible_fit_success": np.nan if flexible is None else flexible.success, "reference_test_mass_retained": float(source_sample.weights[source_evaluation].sum()/source_sample.weights[test_mask].sum()), "target_test_mass_retained": float(target.weights[target_evaluation].sum()/target.weights[test_mask].sum()), "linked_ESS_fraction": interaction.effective_sample_fraction(linked_weights[source_evaluation]), "independent_ESS_fraction": interaction.effective_sample_fraction(independent_weights[source_evaluation]), "projection_unexplained_fraction": projection_unexplained}
        for name, source_values, target_values, reference_valid, target_valid in (
            ("kappa", source_sample.kappa / REFERENCE_K_EFF, target.kappa / REFERENCE_K_EFF, np.ones(len(source_sample.weights), dtype=bool), np.ones(len(target.weights), dtype=bool)),
            ("kappa_prime", source_sample.kappa_prime / REFERENCE_K_EFF**2, target.kappa_prime / REFERENCE_K_EFF**2, np.ones(len(source_sample.weights), dtype=bool), np.ones(len(target.weights), dtype=bool)),
            ("kappa_tau", source_sample.kappa_tau / REFERENCE_K_EFF**2, target.kappa_tau / REFERENCE_K_EFF**2, np.ones(len(source_sample.weights), dtype=bool), np.ones(len(target.weights), dtype=bool)),
        ):
            reference_keep = source_evaluation & reference_valid
            target_keep = target_evaluation & target_valid
            source_values_kept = source_values[reference_keep]
            target_values_kept = target_values[target_keep]
            target_weights = target.weights[target_keep]; target_weights /= target_weights.sum()
            linked_values_weights = linked_weights[reference_keep]; linked_values_weights /= linked_values_weights.sum()
            independent_values_weights = independent_weights[reference_keep]; independent_values_weights /= independent_values_weights.sum()
            flexible_weights = None if flexible_full is None else flexible_full[reference_keep] / flexible_full[reference_keep].sum()
            quality[f"KS_{name}"] = interaction.weighted_ks_distance(target_values_kept, target_weights, source_values_kept, linked_values_weights)
            quality[f"KS_{name}_independent"] = interaction.weighted_ks_distance(target_values_kept, target_weights, source_values_kept, independent_values_weights)
            quality[f"KS_{name}_flexible"] = np.nan if flexible_weights is None else interaction.weighted_ks_distance(target_values_kept, target_weights, source_values_kept, flexible_weights)
            variable_payload[name] = (source_values_kept, linked_values_weights, independent_values_weights, flexible_weights, target_values_kept, target_weights)
        source_linked_features = interaction.local_cell_features(source_sample, source_k_eff, source_expansion_range)
        target_linked_features = interaction.local_cell_features(target, source_k_eff, source_expansion_range)
        local_range_factor = source_expansion_range * source_k_eff
        common_reference_scale = source_k_eff / REFERENCE_K_EFF
        source_local_densities = source_linked_features / local_range_factor * np.array([common_reference_scale**2, common_reference_scale**4])
        target_local_densities = target_linked_features / local_range_factor * np.array([common_reference_scale**2, common_reference_scale**4])
        for feature_index, name in enumerate(("local_k2", "local_k4")):
            source_values_kept = source_local_densities[source_evaluation, feature_index]
            target_values_kept = target_local_densities[target_evaluation, feature_index]
            target_weights = target.weights[target_evaluation]; target_weights /= target_weights.sum()
            linked_values_weights = linked_weights[source_evaluation]; linked_values_weights /= linked_values_weights.sum()
            independent_values_weights = independent_weights[source_evaluation]; independent_values_weights /= independent_values_weights.sum()
            flexible_weights = None if flexible_full is None else flexible_full[source_evaluation] / flexible_full[source_evaluation].sum()
            quality[f"KS_{name}"] = interaction.weighted_ks_distance(target_values_kept, target_weights, source_values_kept, linked_values_weights)
            variable_payload[name] = (source_values_kept, linked_values_weights, independent_values_weights, flexible_weights, target_values_kept, target_weights)
        reconstruction_payload[concentration] = variable_payload
        reconstruction_quality_rows.append(quality)
        worst_ks = max(quality["KS_kappa"], quality["KS_kappa_tau"])
        if (not linked_train.success) or worst_ks > RECONSTRUCTION_KS_WARNING:
            print(f"WARNING {pair.pair_id} linked fit: success={linked_train.success}, largest KS={worst_ks:.3f}")

    reconstruction_ranges = {}
    for name in ("kappa", "kappa_prime", "kappa_tau", "local_k2", "local_k4"):
        bounds = []
        for payload in reconstruction_payload.values():
            reference_values, linked_values_weights, independent_values_weights, flexible_weights, target_values, target_weights = payload[name]
            bounds.append(interaction.weighted_quantile(target_values, [0.005, 0.995], target_weights))
            bounds.append(interaction.weighted_quantile(reference_values, [0.005, 0.995], linked_values_weights))
            bounds.append(interaction.weighted_quantile(reference_values, [0.005, 0.995], independent_values_weights))
        bounds = np.asarray(bounds)
        if name == "local_k2":
            reconstruction_ranges[name] = LOCAL_K2_HISTOGRAM_RANGE
        elif name == "local_k4":
            reconstruction_ranges[name] = LOCAL_K4_HISTOGRAM_RANGE
        elif name == "kappa":
            reconstruction_ranges[name] = (0.0, np.max(bounds[:,1]))
        else:
            limit = np.max(np.abs(bounds)); reconstruction_ranges[name] = (-limit, limit)

    histogram_rows = []
    for concentration, payload in reconstruction_payload.items():
        for name in ("kappa", "kappa_prime", "kappa_tau", "local_k2", "local_k4"):
            reference_values, linked_values_weights, independent_values_weights, flexible_weights, target_values, target_weights = payload[name]
            histogram_bins = LOCAL_K4_HISTOGRAM_BINS if name == "local_k4" else 90
            target_density, edges = np.histogram(target_values, bins=histogram_bins, range=reconstruction_ranges[name], weights=target_weights, density=True)
            linked_density, _ = np.histogram(reference_values, bins=edges, weights=linked_values_weights, density=True)
            independent_density, _ = np.histogram(reference_values, bins=edges, weights=independent_values_weights, density=True)
            flexible_density = np.full_like(linked_density, np.nan) if flexible_weights is None else np.histogram(reference_values, bins=edges, weights=flexible_weights, density=True)[0]
            centers = 0.5*(edges[:-1]+edges[1:])
            pair = pair_context_by_target[concentration]["pair"]
            histogram_rows.extend({**pair.metadata, "concentration_mM": concentration, "variable": name, "coordinate": center, "measured_density": measured_value, "fit_density": linked_value, "linked_density": linked_value, "independent_density": independent_value, "flexible_density": flexible_value} for center,measured_value,linked_value,independent_value,flexible_value in zip(centers,target_density,linked_density,independent_density,flexible_density))
    step_changes = np.asarray([interaction.rescale_dimensionless_coefficient_change(linked_fit_results[pair.target_concentration_mM].coefficients, spectrum_by_salt[pair.source_concentration_mM].k_eff, REFERENCE_K_EFF) for pair in comparison_pairs])
    step_covariances = np.asarray([interaction.rescale_coefficient_covariance(linked_fit_results[pair.target_concentration_mM].covariance, spectrum_by_salt[pair.source_concentration_mM].k_eff, REFERENCE_K_EFF) for pair in comparison_pairs])
    if COMPARISON_MODE == "adjacent":
        cumulative_changes = interaction.cumulative_adjacent_coefficient_changes(comparison_pairs, step_changes, REFERENCE_SALT)
        ordered_salts = sorted(cumulative_changes)
        reference_position = ordered_salts.index(REFERENCE_SALT)
        cumulative_covariances = {}
        for position, salt_value in enumerate(ordered_salts):
            lower, upper = sorted((position, reference_position))
            cumulative_covariances[salt_value] = np.sum(step_covariances[lower:upper], axis=0) if upper > lower else np.zeros((2,2))
    else:
        cumulative_changes = {REFERENCE_SALT: np.zeros(2), **{pair.target_concentration_mM: change for pair, change in zip(comparison_pairs, step_changes)}}
        cumulative_covariances = {REFERENCE_SALT: np.zeros((2,2)), **{pair.target_concentration_mM: covariance for pair, covariance in zip(comparison_pairs, step_covariances)}}
    cumulative_rows = []
    for salt_value, change in sorted(cumulative_changes.items()):
        covariance = cumulative_covariances[salt_value]
        errors = np.sqrt(np.maximum(np.diag(covariance), 0.0))
        cumulative_rows.append({"concentration_mM": salt_value, "reference_concentration_mM": REFERENCE_SALT, "delta_c2_kref": change[0], "delta_c2_kref_se_naive": errors[0], "delta_c4_kref3": change[1], "delta_c4_kref3_se_naive": errors[1], "cov_c2_c2": covariance[0,0], "cov_c2_c4": covariance[0,1], "cov_c4_c4": covariance[1,1], "comparison_mode": COMPARISON_MODE, "covariance_assumption": "adjacent steps treated as independent"})

    distance_q = np.geomspace(0.25 * LOCAL_EXPANSION_RANGE_MULTIPLIER, 12.0, 240)
    physical_distance = distance_q / REFERENCE_K_EFF
    step_kernel_by_pair = {}
    interaction_kernel_rows = []
    step_profile_rows = []
    for pair in comparison_pairs:
        source_k_eff = spectrum_by_salt[pair.source_concentration_mM].k_eff
        linked = linked_fit_results[pair.target_concentration_mM]
        kernel, kernel_mode, ranges, strengths, condition_number = interaction.physical_yukawa_change_from_moments(linked.delta_c2_kref, linked.delta_c4_kref3, source_k_eff, LOCAL_EXPANSION_RANGE_MULTIPLIER, NONLOCAL_KERNEL_MODE, NONLOCAL_TWO_RANGES_KREF)
        step_kernel_by_pair[pair.pair_id] = kernel
        for component, (strength, distance) in enumerate(zip(strengths, ranges)):
            interaction_kernel_rows.append({**pair.metadata, "component": component, "kernel_mode": kernel_mode, "g_delta": strength, "D_delta": distance, "g_delta_over_ksource": strength/source_k_eff, "D_delta_times_ksource": distance*source_k_eff, "condition_number": condition_number, "interpretation": "Delta V parameters, not differences of absolute g or D"})
        step_profile_rows.extend({**pair.metadata, "separation": distance, "separation_times_kref": q_value, "beta_delta_V": value} for distance, q_value, value in zip(physical_distance, distance_q, kernel.evaluate(physical_distance)))
    if COMPARISON_MODE == "adjacent":
        cumulative_kernels = interaction.cumulative_adjacent_yukawa_changes(comparison_pairs, [step_kernel_by_pair[pair.pair_id] for pair in comparison_pairs], REFERENCE_SALT)
    else:
        cumulative_kernels = {REFERENCE_SALT: None, **{pair.target_concentration_mM: step_kernel_by_pair[pair.pair_id] for pair in comparison_pairs}}
    cumulative_profile_rows = []
    for salt_value, kernel in sorted(cumulative_kernels.items()):
        profile = np.zeros_like(physical_distance) if kernel is None else kernel.evaluate(physical_distance)
        cumulative_profile_rows.extend({"concentration_mM": salt_value, "reference_concentration_mM": REFERENCE_SALT, "separation": distance, "separation_times_kref": q_value, "beta_delta_V": value, "comparison_mode": COMPARISON_MODE} for distance, q_value, value in zip(physical_distance, distance_q, profile))

    closure_rows = []
    if RUN_LONG_BASELINE_CLOSURE:
        if COMPARISON_MODE != "adjacent":
            raise RuntimeError("Long-baseline closure requires adjacent primary comparisons")
        baseline_pair = interaction.interaction_comparison_pairs(fit_series, "long_baseline")[0]
        baseline_source = local_samples[baseline_pair.source_concentration_mM]
        baseline_target = local_samples[baseline_pair.target_concentration_mM]
        baseline_k = spectrum_by_salt[baseline_pair.source_concentration_mM].k_eff
        baseline_fit = interaction.fit_linked_maximum_entropy(baseline_source, baseline_target, baseline_k, cell_length=LOCAL_EXPANSION_RANGE_MULTIPLIER/baseline_k)
        local_means = np.asarray([linked_fit_results[pair.target_concentration_mM].coefficients for pair in comparison_pairs])
        local_covariances = np.asarray([linked_fit_results[pair.target_concentration_mM].covariance for pair in comparison_pairs])
        source_scales = np.asarray([spectrum_by_salt[pair.source_concentration_mM].k_eff for pair in comparison_pairs])
        summed_mean, summed_sd, summed_valid = interaction.sampled_yukawa_change_profile(local_means, local_covariances, source_scales, physical_distance, LOCAL_EXPANSION_RANGE_MULTIPLIER, CLOSURE_UNCERTAINTY_SAMPLES, 71403, NONLOCAL_KERNEL_MODE, NONLOCAL_TWO_RANGES_KREF)
        direct_mean, direct_sd, direct_valid = interaction.sampled_yukawa_change_profile(baseline_fit.coefficients, baseline_fit.covariance[None,:,:], np.array([baseline_k]), physical_distance, LOCAL_EXPANSION_RANGE_MULTIPLIER, CLOSURE_UNCERTAINTY_SAMPLES, 81403, NONLOCAL_KERNEL_MODE, NONLOCAL_TWO_RANGES_KREF)
        difference = direct_mean - summed_mean
        difference_sd = np.sqrt(direct_sd**2 + summed_sd**2)
        closure_rows = [{"separation": distance, "separation_times_kref": q_value, "direct_beta_delta_V": direct_value, "direct_1sd": direct_error, "adjacent_sum_beta_delta_V": summed_value, "adjacent_sum_1sd": summed_error, "difference": delta, "difference_1sd": delta_error, "z_score": delta/delta_error if delta_error > 0 else np.nan, "direct_valid_draws": direct_valid, "adjacent_valid_draws": summed_valid} for distance, q_value, direct_value, direct_error, summed_value, summed_error, delta, delta_error in zip(physical_distance, distance_q, direct_mean, direct_sd, summed_mean, summed_sd, difference, difference_sd)]

    coefficient_rows.sort(key=lambda row: float(row["concentration_mM"]))
    independent_rows.sort(key=lambda row: float(row["concentration_mM"]))
    invariant_moment_rows.sort(key=lambda row: float(row["concentration_mM"]))
    structure_correlation_rows.sort(key=lambda row: float(row["concentration_mM"]))
    with (OUTPUT_DIR / "effective_local_cell_coefficients.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(coefficient_rows[0]))
        writer.writeheader(); writer.writerows(coefficient_rows)
    with (OUTPUT_DIR / "independent_fourth_order_coefficients.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(independent_rows[0])); writer.writeheader(); writer.writerows(independent_rows)
    with (OUTPUT_DIR / "local_invariant_moments.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(invariant_moment_rows[0])); writer.writeheader(); writer.writerows(invariant_moment_rows)
    with (OUTPUT_DIR / "local_structure_correlations.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(structure_correlation_rows[0])); writer.writeheader(); writer.writerows(structure_correlation_rows)
    with (OUTPUT_DIR / "local_structure_histograms.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(local_structure_histogram_rows[0])); writer.writeheader(); writer.writerows(local_structure_histogram_rows)
    with (OUTPUT_DIR / "local_reconstruction_histograms.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(histogram_rows[0]))
        writer.writeheader(); writer.writerows(histogram_rows)
    with (OUTPUT_DIR / "local_reconstruction_quality.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(reconstruction_quality_rows[0]))
        writer.writeheader(); writer.writerows(reconstruction_quality_rows)
    extra_artifacts = {"cumulative_local_coefficients.csv": cumulative_rows, "stepwise_interaction_change_kernels.csv": interaction_kernel_rows, "stepwise_interaction_change_profiles.csv": step_profile_rows, "cumulative_interaction_change_profiles.csv": cumulative_profile_rows}
    if closure_rows: extra_artifacts["long_baseline_interaction_closure.csv"] = closure_rows
    for filename, rows in extra_artifacts.items():
        with (OUTPUT_DIR / filename).open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(rows[0])); writer.writeheader(); writer.writerows(rows)
    print(f"saved {len(comparison_pairs)} {COMPARISON_MODE} fits, cumulative changes, interaction profiles, and distribution checks")
else:
    print("local inference skipped; existing fit artifacts are unchanged")

WARNING 0mM_to_1mM: K2,K4 leave 99.2% of the fitted log-density variation unexplained
WARNING 1mM_to_10mM: K2,K4 leave 96.9% of the fitted log-density variation unexplained
WARNING 10mM_to_24mM: K2,K4 leave 93.1% of the fitted log-density variation unexplained
WARNING 24mM_to_40mM: K2,K4 leave 96.2% of the fitted log-density variation unexplained
WARNING 40mM_to_55mM: K2,K4 leave 84.3% of the fitted log-density variation unexplained
WARNING 55mM_to_70mM: K2,K4 leave 99.7% of the fitted log-density variation unexplained
WARNING 55mM_to_70mM linked fit: success=True, largest KS=0.122
WARNING 70mM_to_100mM: K2,K4 leave 95.0% of the fitted log-density variation unexplained
saved 7 adjacent fits, cumulative changes, interaction profiles, and distribution checks


## Optional checks

These checks can take a long time and are off by default.

In [4]:
expansion_range_rows = []
if RUN_EXPANSION_RANGE_SENSITIVITY:
    for pair in comparison_pairs:
        context = pair_context_by_target[pair.target_concentration_mM]
        for length, result in interaction.local_expansion_range_sensitivity(context["source_sample"], context["target_sample"], context["source_k_eff"], np.array([0.75, 1.0, 1.25]) / context["source_k_eff"]):
            expansion_range_rows.append({**pair.metadata, "concentration_mM": pair.target_concentration_mM, "expansion_range_times_ksource": length * context["source_k_eff"], "delta_c2_ksource": result.delta_c2_kref, "delta_c4_ksource3": result.delta_c4_kref3})
    with (OUTPUT_DIR / "local_expansion_range_sensitivity.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(expansion_range_rows[0]))
        writer.writeheader(); writer.writerows(expansion_range_rows)
print("local expansion-range sensitivity rows:", len(expansion_range_rows))

local expansion-range sensitivity rows: 0


In [5]:
replicate_rows = []
if RUN_REPLICATE_UNCERTAINTY:
    replicate_samples = [interaction.sample_local_geometry_series(fit_series, sample_power=REPLICATE_SAMPLE_POWER, seed=seed, aniso=ANISOTROPIC_GEOMETRY, H_by_concentration=hencky_by_salt) for seed in (101, 211, 307, 401, 503, 601)]
    for pair in comparison_pairs:
        source_k_eff = spectrum_by_salt[pair.source_concentration_mM].k_eff
        result = interaction.bootstrap_local_cell_replicates([samples[pair.source_concentration_mM] for samples in replicate_samples], [samples[pair.target_concentration_mM] for samples in replicate_samples], source_k_eff, LOCAL_EXPANSION_RANGE_MULTIPLIER/source_k_eff, bootstrap_samples=REPLICATE_BOOTSTRAP_SAMPLES, seed=7103)
        reported_mean = interaction.rescale_dimensionless_coefficient_change(result.mean, source_k_eff, REFERENCE_K_EFF)
        reported_covariance = interaction.rescale_coefficient_covariance(result.covariance, source_k_eff, REFERENCE_K_EFF)
        replicate_rows.append({**pair.metadata, "concentration_mM": pair.target_concentration_mM, "delta_c2_mean": reported_mean[0], "delta_c4_mean": reported_mean[1], "cov_c2_c2": reported_covariance[0,0], "cov_c2_c4": reported_covariance[0,1], "cov_c4_c4": reported_covariance[1,1]})
    with (OUTPUT_DIR / "replicate_coefficient_uncertainty.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(replicate_rows[0]))
        writer.writeheader(); writer.writerows(replicate_rows)
print("replicate uncertainty rows:", len(replicate_rows))

replicate uncertainty rows: 0


## Optional contour traces

Existing traces are reused when possible. Build new traces only when the nonlocal or contour calculation needs them.

In [6]:
nonlocal_trace_salts = set(spectrum_by_salt) if (RUN_NONLOCAL_DEPENDENCE or RUN_NONLOCAL_CORRECTION) else set()
required_source_salts = nonlocal_trace_salts | ({pair.source_concentration_mM for pair in comparison_pairs} if RUN_CONTOUR_VALIDATION else set())
required_target_salts = {pair.target_concentration_mM for pair in comparison_pairs} if RUN_CONTOUR_VALIDATION else set()
required_trace_salts = required_source_salts | required_target_salts
traces_by_salt = {}
trace_manifest_rows = []
for spectrum in fit_series:
    concentration = spectrum.concentration_mM
    if concentration not in required_trace_salts:
        continue
    label = f"{concentration:g}".replace(".", "p")
    if concentration == REFERENCE_SALT:
        cache = ISOTROPIC_TRACE_DIR / "reference_isotropic_trace.npz"
        build_missing = BUILD_REFERENCE_TRACE
    else:
        cache = ISOTROPIC_TRACE_DIR / f"target_{label}mM_isotropic_trace.npz"
        build_missing = BUILD_TARGET_TRACES
    trace_base = interaction.load_trace_cache(cache, TRACE_CONFIG, spectrum)
    if trace_base is None and build_missing:
        trace_base = interaction.build_isotropic_trace(PROJECT_ROOT, spectrum, TRACE_CONFIG)
        interaction.save_trace_cache(cache, trace_base, TRACE_CONFIG, spectrum)
    trace = trace_base
    analysis_cache = cache
    if ANISOTROPIC_GEOMETRY and trace_base is not None:
        trace = interaction.transform_contour_trace(trace_base, hencky_by_salt[concentration])
        analysis_cache = OUTPUT_DIR / f"trace_{label}mM_aniso.npz"
        interaction.save_trace_cache(analysis_cache, trace, TRACE_CONFIG, spectrum)
    if trace is not None:
        traces_by_salt[concentration] = trace
    trace_manifest_rows.append({"concentration_mM": concentration, "role": "pair_source" if concentration in required_source_salts else "pair_target", "cache": str(analysis_cache), "available": trace is not None, "k_eff": spectrum.k_eff})
reference_trace = traces_by_salt.get(REFERENCE_SALT)
target_traces = {salt: trace for salt, trace in traces_by_salt.items() if salt != REFERENCE_SALT}
if trace_manifest_rows:
    with (OUTPUT_DIR / "trace_manifest.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(trace_manifest_rows[0]))
        writer.writeheader(); writer.writerows(trace_manifest_rows)
missing_trace_salts = sorted(required_trace_salts - set(traces_by_salt))
print("traces available:", len(traces_by_salt), "; missing:", missing_trace_salts)

traces available: 8 ; missing: []


## Optional checks required by the note

The first check asks whether the distant part of the interaction varies systematically with the local shape $X=\{\kappa,\kappa',\kappa\tau\}$. The second compares the fast local-expansion coefficients with coefficients obtained by integrating $K_2$ and $K_4$ along traced contours.

In [7]:
nonlocal_dependence_rows = []
dependence_by_source = {}
series_requires_shape_correction = False
if RUN_NONLOCAL_DEPENDENCE or RUN_NONLOCAL_CORRECTION:
    nonlocal_salts = sorted(spectrum_by_salt)
    missing = sorted(set(nonlocal_salts) - set(traces_by_salt))
    if missing:
        raise RuntimeError(f"Build traces for every salt condition before distant-interaction work: {missing}")
    for source_salt in nonlocal_salts:
        source_trace = traces_by_salt[source_salt]
        source_k_eff = spectrum_by_salt[source_salt].k_eff
        nonlocal_config = interaction.NonlocalConfig.from_dimensionless(source_k_eff, q_min=LOCAL_EXPANSION_RANGE_MULTIPLIER, q_max=8.0)
        representative_potential = interaction.YukawaPotential(source_k_eff, 1.0/source_k_eff)
        table = interaction.absolute_nonlocal_energy_density(source_trace, representative_potential, nonlocal_config)
        dependence = interaction.assess_nonlocal_dependence(table)
        dependence_by_source[source_salt] = dependence
        nonlocal_dependence_rows.append({"source_concentration_mM": source_salt, "range_times_ksource": 1.0, "svd_fraction": dependence.svd_retained_fraction, "heldout_R2": dependence.mean_r2, "heldout_R2_se": dependence.standard_error, "depends_on_local_shape": dependence.dependent})
    with (OUTPUT_DIR / "nonlocal_dependence.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(nonlocal_dependence_rows[0]))
        writer.writeheader(); writer.writerows(nonlocal_dependence_rows)
    dependent_salts = [salt for salt, result in dependence_by_source.items() if result.dependent]
    series_requires_shape_correction = bool(dependent_salts)
    if NONLOCAL_SHAPE_CORRECTION_OVERRIDE is True: series_requires_shape_correction = True
    if NONLOCAL_SHAPE_CORRECTION_OVERRIDE is False: series_requires_shape_correction = False
    print("salt conditions with shape-dependent distant interaction:", dependent_salts)
    print("apply the correction to every comparison pair:", series_requires_shape_correction)

nonlocal_correction_rows = []
nonlocal_corrected_histogram_rows = []
nonlocal_corrected_quality_rows = []
nonlocal_weight_payload = {}
nonlocal_corrected_fit_by_pair = {}
nonlocal_corrected_kernel_by_pair = {}
nonlocal_corrected_step_profile_rows = []
nonlocal_corrected_cumulative_profile_rows = []
nonlocal_corrected_closure_rows = []
if RUN_NONLOCAL_CORRECTION and not series_requires_shape_correction:
    print("No salt condition requires the distant-interaction correction; no comparison pair is corrected.")
if RUN_NONLOCAL_CORRECTION and series_requires_shape_correction:
    if not RUN_RANDOMWAVE_SAMPLING:
        raise RuntimeError("Run the local inference before the distant-interaction correction")
    for pair in comparison_pairs:
        concentration = pair.target_concentration_mM
        context = pair_context_by_target[concentration]
        source_sample = context["source_sample"]; target = context["target_sample"]
        source_k_eff = context["source_k_eff"]; source_expansion_range = context["expansion_range"]
        source_trace = traces_by_salt[pair.source_concentration_mM]
        local_fit = linked_fit_results[concentration]
        nonlocal_config = interaction.NonlocalConfig.from_dimensionless(source_k_eff, q_min=LOCAL_EXPANSION_RANGE_MULTIPLIER, q_max=8.0)
        shape_correction_allowed = True
        force_shape_correction = True
        corrected = interaction.fit_nonlocal_corrected_maximum_entropy(source_trace, source_sample, target, source_k_eff, local_fit.delta_c2_kref, local_fit.delta_c4_kref3, nonlocal_config, cell_length=source_expansion_range, maximum_iterations=NONLOCAL_MAX_ITERATIONS, damping=NONLOCAL_DAMPING, kernel_mode=NONLOCAL_KERNEL_MODE, two_ranges_kref=NONLOCAL_TWO_RANGES_KREF, shape_dependence_required=shape_correction_allowed, force_shape_dependence=force_shape_correction)
        nonlocal_corrected_fit_by_pair[pair.pair_id] = corrected.corrected_fit
        corrected_kernel, _, _, _, _ = interaction.physical_yukawa_change_from_moments(corrected.corrected_fit.delta_c2_kref, corrected.corrected_fit.delta_c4_kref3, source_k_eff, LOCAL_EXPANSION_RANGE_MULTIPLIER, NONLOCAL_KERNEL_MODE, NONLOCAL_TWO_RANGES_KREF)
        nonlocal_corrected_kernel_by_pair[pair.pair_id] = corrected_kernel
        nonlocal_corrected_step_profile_rows.extend({**pair.metadata, "separation": distance, "separation_times_kref": q_value, "beta_delta_V": value} for distance, q_value, value in zip(physical_distance, distance_q, corrected_kernel.evaluate(physical_distance)))
        corrected_train = interaction.fit_linked_maximum_entropy(source_sample, target, source_k_eff, cell_length=source_expansion_range, reference_mask=train_mask, target_mask=train_mask, initial=corrected.corrected_fit.coefficients, reference_beta_delta_f_nl=corrected.beta_delta_free_energy)
        source_features = interaction.local_cell_features(source_sample, source_k_eff, source_expansion_range)
        target_features = interaction.local_cell_features(target, source_k_eff, source_expansion_range)
        source_support = np.all((source_features >= corrected_train.feature_bounds[:,0]) & (source_features <= corrected_train.feature_bounds[:,1]), axis=1)
        target_support = np.all((target_features >= corrected_train.feature_bounds[:,0]) & (target_features <= corrected_train.feature_bounds[:,1]), axis=1)
        source_keep = test_mask & source_support; target_keep = test_mask & target_support
        predicted = interaction.maximum_entropy_reweighted_weights(source_sample, corrected_train, source_k_eff, source_expansion_range, mask=source_keep, beta_delta_f_nl=corrected.beta_delta_free_energy)
        quality = {**pair.metadata, "concentration_mM": concentration, "KS_kappa": np.nan, "KS_kappa_prime": np.nan, "KS_kappa_tau": np.nan}
        for name, source_values, target_values in (("kappa", source_sample.kappa/REFERENCE_K_EFF, target.kappa/REFERENCE_K_EFF), ("kappa_prime", source_sample.kappa_prime/REFERENCE_K_EFF**2, target.kappa_prime/REFERENCE_K_EFF**2), ("kappa_tau", source_sample.kappa_tau/REFERENCE_K_EFF**2, target.kappa_tau/REFERENCE_K_EFF**2)):
            source_values = source_values[source_keep]; target_values = target_values[target_keep]
            predicted_weights = predicted[source_keep]; predicted_weights /= predicted_weights.sum()
            measured_weights = target.weights[target_keep]; measured_weights /= measured_weights.sum()
            quality[f"KS_{name}"] = interaction.weighted_ks_distance(target_values, measured_weights, source_values, predicted_weights)
            measured_density, edges = np.histogram(target_values, bins=90, range=reconstruction_ranges[name], weights=measured_weights, density=True)
            corrected_density, _ = np.histogram(source_values, bins=edges, weights=predicted_weights, density=True)
            centers = 0.5*(edges[:-1]+edges[1:])
            nonlocal_corrected_histogram_rows.extend({**pair.metadata, "concentration_mM": concentration, "variable": name, "coordinate": center, "measured_density": measured, "fit_density": fitted} for center, measured, fitted in zip(centers, measured_density, corrected_density))
        nonlocal_corrected_quality_rows.append(quality)
        local_reported = interaction.rescale_dimensionless_coefficient_change(local_fit.coefficients, source_k_eff, REFERENCE_K_EFF)
        corrected_reported = interaction.rescale_dimensionless_coefficient_change(corrected.corrected_fit.coefficients, source_k_eff, REFERENCE_K_EFF)
        corrected_covariance = interaction.rescale_coefficient_covariance(corrected.corrected_fit.covariance, source_k_eff, REFERENCE_K_EFF)
        errors = np.sqrt(np.maximum(np.diag(corrected_covariance), 0.0))
        beta_mean = float(np.sum(source_sample.weights*corrected.beta_delta_free_energy))
        beta_std = float(np.sqrt(np.sum(source_sample.weights*(corrected.beta_delta_free_energy-beta_mean)**2)))
        nonlocal_correction_rows.append({**pair.metadata, "concentration_mM": concentration, "reference_concentration_mM": pair.source_concentration_mM, "delta_c2_kref_local": local_reported[0], "delta_c4_kref3_local": local_reported[1], "delta_c2_kref_corrected": corrected_reported[0], "delta_c2_kref_corrected_se_naive": errors[0], "delta_c4_kref3_corrected": corrected_reported[1], "delta_c4_kref3_corrected_se_naive": errors[1], "delta_c2_ksource_corrected": corrected.corrected_fit.delta_c2_kref, "delta_c4_ksource3_corrected": corrected.corrected_fit.delta_c4_kref3, "kernel_mode": corrected.kernel_mode, "kernel_ranges_ksource": ";".join(f"{value:.12g}" for value in corrected.kernel_ranges_kref), "kernel_strengths_over_ksource": ";".join(f"{value:.12g}" for value in corrected.kernel_strengths_over_kref), "kernel_condition_number": corrected.kernel_condition_number, "relative_heldout_R2": corrected.dependence.mean_r2, "relative_heldout_R2_se": corrected.dependence.standard_error, "source_test_requires_shape_correction": corrected.shape_dependence_required, "beta_delta_f_mean": beta_mean, "beta_delta_f_std": beta_std, "corrected_base_ESS_fraction": interaction.effective_sample_fraction(corrected.corrected_reference_weights), "fixed_point_converged": corrected.converged})
        key = pair.pair_id.replace(".", "p")
        nonlocal_weight_payload[f"{key}_beta_delta_f_nl"] = corrected.beta_delta_free_energy
        nonlocal_weight_payload[f"{key}_corrected_source_weights"] = corrected.corrected_reference_weights
    if COMPARISON_MODE == "adjacent":
        corrected_cumulative_kernels = interaction.cumulative_adjacent_yukawa_changes(comparison_pairs, [nonlocal_corrected_kernel_by_pair[pair.pair_id] for pair in comparison_pairs], REFERENCE_SALT)
    else:
        corrected_cumulative_kernels = {REFERENCE_SALT: None, **{pair.target_concentration_mM: nonlocal_corrected_kernel_by_pair[pair.pair_id] for pair in comparison_pairs}}
    for salt_value, kernel in sorted(corrected_cumulative_kernels.items()):
        profile = np.zeros_like(physical_distance) if kernel is None else kernel.evaluate(physical_distance)
        nonlocal_corrected_cumulative_profile_rows.extend({"concentration_mM": salt_value, "reference_concentration_mM": REFERENCE_SALT, "separation": distance, "separation_times_kref": q_value, "beta_delta_V": value, "comparison_mode": COMPARISON_MODE} for distance, q_value, value in zip(physical_distance, distance_q, profile))
    if RUN_LONG_BASELINE_CLOSURE:
        if COMPARISON_MODE != "adjacent":
            raise RuntimeError("Corrected long-baseline closure requires adjacent primary comparisons")
        baseline_pair = interaction.interaction_comparison_pairs(fit_series, "long_baseline")[0]
        baseline_source = local_samples[baseline_pair.source_concentration_mM]
        baseline_target = local_samples[baseline_pair.target_concentration_mM]
        baseline_k = spectrum_by_salt[baseline_pair.source_concentration_mM].k_eff
        baseline_expansion_range = LOCAL_EXPANSION_RANGE_MULTIPLIER / baseline_k
        baseline_local = interaction.fit_linked_maximum_entropy(baseline_source, baseline_target, baseline_k, cell_length=baseline_expansion_range)
        baseline_config = interaction.NonlocalConfig.from_dimensionless(baseline_k, q_min=LOCAL_EXPANSION_RANGE_MULTIPLIER, q_max=8.0)
        baseline_corrected = interaction.fit_nonlocal_corrected_maximum_entropy(traces_by_salt[baseline_pair.source_concentration_mM], baseline_source, baseline_target, baseline_k, baseline_local.delta_c2_kref, baseline_local.delta_c4_kref3, baseline_config, cell_length=baseline_expansion_range, maximum_iterations=NONLOCAL_MAX_ITERATIONS, damping=NONLOCAL_DAMPING, kernel_mode=NONLOCAL_KERNEL_MODE, two_ranges_kref=NONLOCAL_TWO_RANGES_KREF, shape_dependence_required=True, force_shape_dependence=True)
        corrected_means = np.asarray([nonlocal_corrected_fit_by_pair[pair.pair_id].coefficients for pair in comparison_pairs])
        corrected_covariances = np.asarray([nonlocal_corrected_fit_by_pair[pair.pair_id].covariance for pair in comparison_pairs])
        source_scales = np.asarray([spectrum_by_salt[pair.source_concentration_mM].k_eff for pair in comparison_pairs])
        summed_mean, summed_sd, summed_valid = interaction.sampled_yukawa_change_profile(corrected_means, corrected_covariances, source_scales, physical_distance, LOCAL_EXPANSION_RANGE_MULTIPLIER, CLOSURE_UNCERTAINTY_SAMPLES, 91403, NONLOCAL_KERNEL_MODE, NONLOCAL_TWO_RANGES_KREF)
        direct_mean, direct_sd, direct_valid = interaction.sampled_yukawa_change_profile(baseline_corrected.corrected_fit.coefficients, baseline_corrected.corrected_fit.covariance[None,:,:], np.array([baseline_k]), physical_distance, LOCAL_EXPANSION_RANGE_MULTIPLIER, CLOSURE_UNCERTAINTY_SAMPLES, 101403, NONLOCAL_KERNEL_MODE, NONLOCAL_TWO_RANGES_KREF)
        difference = direct_mean - summed_mean; difference_sd = np.sqrt(direct_sd**2 + summed_sd**2)
        nonlocal_corrected_closure_rows = [{"separation": distance, "separation_times_kref": q_value, "direct_beta_delta_V": direct_value, "direct_1sd": direct_error, "adjacent_sum_beta_delta_V": summed_value, "adjacent_sum_1sd": summed_error, "difference": delta, "difference_1sd": delta_error, "z_score": delta/delta_error if delta_error > 0 else np.nan, "direct_valid_draws": direct_valid, "adjacent_valid_draws": summed_valid} for distance, q_value, direct_value, direct_error, summed_value, summed_error, delta, delta_error in zip(physical_distance, distance_q, direct_mean, direct_sd, summed_mean, summed_sd, difference, difference_sd)]
    corrected_artifacts = {"nonlocal_corrected_coefficients.csv": nonlocal_correction_rows, "nonlocal_corrected_reconstruction_histograms.csv": nonlocal_corrected_histogram_rows, "nonlocal_corrected_reconstruction_quality.csv": nonlocal_corrected_quality_rows, "nonlocal_corrected_stepwise_interaction_change_profiles.csv": nonlocal_corrected_step_profile_rows, "nonlocal_corrected_cumulative_interaction_change_profiles.csv": nonlocal_corrected_cumulative_profile_rows}
    if nonlocal_corrected_closure_rows:
        corrected_artifacts["nonlocal_corrected_long_baseline_interaction_closure.csv"] = nonlocal_corrected_closure_rows
    for filename, rows in corrected_artifacts.items():
        with (OUTPUT_DIR / filename).open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(rows[0])); writer.writeheader(); writer.writerows(rows)
    np.savez_compressed(OUTPUT_DIR / "nonlocal_corrected_reference_weights.npz", **nonlocal_weight_payload)
    print("distant-interaction corrected transitions:", len(nonlocal_correction_rows))

contour_validation_rows = []
contour_reconstruction_histogram_rows = []
contour_reconstruction_quality_rows = []
if RUN_CONTOUR_VALIDATION:
    needed = {value for pair in comparison_pairs for value in (pair.source_concentration_mM, pair.target_concentration_mM)}
    missing = sorted(needed - set(traces_by_salt))
    if missing:
        raise RuntimeError(f"Build the required contour traces first; missing conditions: {missing}")
    for pair in comparison_pairs:
        source_trace = traces_by_salt[pair.source_concentration_mM]
        target_trace = traces_by_salt[pair.target_concentration_mM]
        source_k_eff = spectrum_by_salt[pair.source_concentration_mM].k_eff
        contour_block_length = CONTOUR_BLOCK_RANGE_MULTIPLIER / source_k_eff
        source_block_states = interaction.trace_local_state_blocks(source_trace, contour_block_length)
        target_block_states = interaction.trace_local_state_blocks(target_trace, contour_block_length)
        source_observables = np.asarray([interaction.contour_observables(state[:,0], state[:,1], state[:,2], np.arange(len(state))*source_trace.physical_spacing) for state in source_block_states])
        target_observables = np.asarray([interaction.contour_observables(state[:,0], state[:,1], state[:,2], np.arange(len(state))*target_trace.physical_spacing) for state in target_block_states])
        contour_fit = interaction.fit_contour_maximum_entropy(source_observables, target_observables, np.ones(len(source_observables)), np.ones(len(target_observables)), source_k_eff, support_quantiles=CONTOUR_SUPPORT_QUANTILES)
        heldout = interaction.fit_contour_heldout_maximum_entropy(source_observables, target_observables, source_k_eff, support_quantiles=CONTOUR_SUPPORT_QUANTILES, seed=48031 + pair.step_index)
        contour_train_fit = heldout.fit
        source_train = heldout.reference_train
        target_train = heldout.target_train
        if heldout.split_attempt:
            print(f"{pair.pair_id}: used held-out split attempt {heldout.split_attempt + 1}")
        predicted_contour_weights = interaction.contour_maximum_entropy_weights(source_observables, contour_train_fit, source_k_eff, mask=~source_train)
        target_contour_keep = heldout.target_test_keep
        measured_contour_weights = target_contour_keep.astype(float); measured_contour_weights /= measured_contour_weights.sum()
        predicted_state, predicted_point_weights = interaction.local_state_point_weights(source_block_states, predicted_contour_weights, REFERENCE_K_EFF)
        measured_state, measured_point_weights = interaction.local_state_point_weights(target_block_states, measured_contour_weights, REFERENCE_K_EFF)
        contour_quality = {**pair.metadata, "concentration_mM": pair.target_concentration_mM, "heldout_split_attempt": heldout.split_attempt, "heldout_source_blocks": int(np.count_nonzero(predicted_contour_weights)), "heldout_target_blocks": int(np.count_nonzero(target_contour_keep))}
        predicted_variables = {"kappa": predicted_state[:,0], "kappa_prime": predicted_state[:,1], "kappa_tau": predicted_state[:,2], "local_k2": predicted_state[:,0]**2, "local_k4": (9/64)*predicted_state[:,0]**4 - predicted_state[:,1]**2/24 - predicted_state[:,2]**2/24}
        measured_variables = {"kappa": measured_state[:,0], "kappa_prime": measured_state[:,1], "kappa_tau": measured_state[:,2], "local_k2": measured_state[:,0]**2, "local_k4": (9/64)*measured_state[:,0]**4 - measured_state[:,1]**2/24 - measured_state[:,2]**2/24}
        for variable in ("kappa", "kappa_prime", "kappa_tau", "local_k2", "local_k4"):
            predicted_values = predicted_variables[variable]; measured_values = measured_variables[variable]
            contour_quality[f"KS_{variable}"] = interaction.weighted_ks_distance(measured_values, measured_point_weights, predicted_values, predicted_point_weights)
            display_range = LOCAL_K2_HISTOGRAM_RANGE if variable == "local_k2" else LOCAL_K4_HISTOGRAM_RANGE if variable == "local_k4" else reconstruction_ranges[variable]
            histogram_bins = LOCAL_K4_HISTOGRAM_BINS if variable == "local_k4" else 90
            measured_density, edges = np.histogram(measured_values, bins=histogram_bins, range=display_range, weights=measured_point_weights, density=True)
            predicted_density, _ = np.histogram(predicted_values, bins=edges, weights=predicted_point_weights, density=True)
            centers = 0.5*(edges[:-1]+edges[1:])
            contour_reconstruction_histogram_rows.extend({**pair.metadata, "concentration_mM": pair.target_concentration_mM, "variable": variable, "coordinate": center, "measured_density": measured, "fit_density": fitted, "normalization": "common_kref", "normalization_k_ref": REFERENCE_K_EFF} for center,measured,fitted in zip(centers,measured_density,predicted_density))
        contour_reconstruction_quality_rows.append(contour_quality)
        cell_fit = linked_fit_results[pair.target_concentration_mM] if RUN_RANDOMWAVE_SAMPLING else interaction.fit_linked_maximum_entropy(local_samples[pair.source_concentration_mM], local_samples[pair.target_concentration_mM], source_k_eff, LOCAL_EXPANSION_RANGE_MULTIPLIER/source_k_eff)
        contour_reported = interaction.rescale_dimensionless_coefficient_change(contour_fit.coefficients, source_k_eff, REFERENCE_K_EFF)
        cell_reported = interaction.rescale_dimensionless_coefficient_change(cell_fit.coefficients, source_k_eff, REFERENCE_K_EFF)
        common_scale = source_k_eff / REFERENCE_K_EFF
        randomwave_target_density = cell_fit.target_moments / LOCAL_EXPANSION_RANGE_MULTIPLIER * np.array([common_scale**2, common_scale**4])
        randomwave_fitted_density = cell_fit.fitted_moments / LOCAL_EXPANSION_RANGE_MULTIPLIER * np.array([common_scale**2, common_scale**4])
        contour_target_density = contour_fit.target_moments / CONTOUR_BLOCK_RANGE_MULTIPLIER * np.array([common_scale**2, common_scale**4])
        contour_fitted_density = contour_fit.fitted_moments / CONTOUR_BLOCK_RANGE_MULTIPLIER * np.array([common_scale**2, common_scale**4])
        contour_validation_rows.append({**pair.metadata, "concentration_mM": pair.target_concentration_mM, "delta_c2_kref_contour": contour_reported[0], "delta_c4_kref3_contour": contour_reported[1], "delta_c2_kref_local": cell_reported[0], "delta_c4_kref3_local": cell_reported[1], "randomwave_target_k2_density_over_kref2": randomwave_target_density[0], "randomwave_fitted_k2_density_over_kref2": randomwave_fitted_density[0], "randomwave_target_k4_density_over_kref4": randomwave_target_density[1], "randomwave_fitted_k4_density_over_kref4": randomwave_fitted_density[1], "contour_target_k2_density_over_kref2": contour_target_density[0], "contour_fitted_k2_density_over_kref2": contour_fitted_density[0], "contour_target_k4_density_over_kref4": contour_target_density[1], "contour_fitted_k4_density_over_kref4": contour_fitted_density[1], "target_mean_K2_over_ksource": contour_fit.target_moments[0], "fitted_mean_K2_over_ksource": contour_fit.fitted_moments[0], "mean_residual_K2_over_ksource": contour_fit.fitted_moments[0] - contour_fit.target_moments[0], "target_mean_K4_over_ksource3": contour_fit.target_moments[1], "fitted_mean_K4_over_ksource3": contour_fit.fitted_moments[1], "mean_residual_K4_over_ksource3": contour_fit.fitted_moments[1] - contour_fit.target_moments[1], "KS_kappa": contour_quality["KS_kappa"], "KS_kappa_prime": contour_quality["KS_kappa_prime"], "KS_kappa_tau": contour_quality["KS_kappa_tau"], "KS_local_k2": contour_quality["KS_local_k2"], "KS_local_k4": contour_quality["KS_local_k4"], "heldout_split_attempt": heldout.split_attempt, "heldout_source_blocks": contour_quality["heldout_source_blocks"], "heldout_target_blocks": contour_quality["heldout_target_blocks"]})
    with (OUTPUT_DIR / "contour_validation.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(contour_validation_rows[0]))
        writer.writeheader(); writer.writerows(contour_validation_rows)
    with (OUTPUT_DIR / "contour_reconstruction_histograms.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(contour_reconstruction_histogram_rows[0])); writer.writeheader(); writer.writerows(contour_reconstruction_histogram_rows)
    with (OUTPUT_DIR / "contour_reconstruction_quality.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(contour_reconstruction_quality_rows[0])); writer.writeheader(); writer.writerows(contour_reconstruction_quality_rows)
print("contour comparisons:", len(contour_validation_rows))

salt conditions with shape-dependent distant interaction: [24.0, 100.0]
apply the correction to every comparison pair: True
distant-interaction corrected transitions: 7
contour comparisons: 0


## Optional trace convergence

This expensive check is available for the final production run and is off by default.

In [8]:
trace_convergence_rows = []
if RUN_TRACE_CONVERGENCE:
    configurations = [interaction.TraceConfig(grid_size=64, num_modes=modes, q_spacing=spacing) for modes in (128,256,512) for spacing in (0.12,0.08,0.05)]
    trace_convergence_rows = interaction.run_trace_convergence_sweep(PROJECT_ROOT, reference_spectrum, configurations, metric=lambda trace: {"mean_kappa_over_keff": np.mean(np.vstack(interaction.trace_local_state(trace, normalize=True))[:,0])})
    with (OUTPUT_DIR / "trace_convergence.csv").open("w", newline="", encoding="utf-8") as handle:
        fieldnames = sorted({key for row in trace_convergence_rows for key in row})
        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader(); writer.writerows(trace_convergence_rows)
print("trace convergence rows:", len(trace_convergence_rows))

trace convergence rows: 0
